# The racing line

**The centreline is not the fast line, but a new line breaks your speed tuning.**

Unit 8 measured the car: the steering gain (65.7° measured against the 70° the agent assumed; the agent rounds it to 66°) and the grip of every section, from limit probes.
Its agent plans with a fraction of the measured grip, `MU_FRACTION = 0.56`, frozen one grid step inside the training family's edge of 0.58, and it transferred to the unseen `test-circuit-seed2026` clean.
It still drives the **centreline**, the middle of the road.
A racing driver does not: they use the width of the track to make every corner gentler.

Here is Unit 8's agent next to this unit's, `agents/unit-09.py`, which drives a **racing line**, on `test-circuit` (`carla_like(noise=True)`, seed 0):

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_08, UNIT_09 = AGENTS / "unit-08.py", AGENTS / "unit-09.py"
workdir = Path(mkdtemp(prefix="roar-unit-09-"))
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the cross-track error

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per track.
# agents/unit-08.py and unit-09.py ship test-circuit's; another track's values are written in with write_variant.
MEASURED_MU_G = {
    "test-circuit": (7.89, 9.72, 8.65, 7.87, 8.30, 8.14),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
}


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, track="test-circuit", seed=0, max_seconds=450.0):
    # Every race of this lesson: the carla-like car with its noise, one seed.
    return run_race(source, track, car=carla_like(noise=True), seed=seed, max_seconds=max_seconds)


centre_race = race(UNIT_08)  # Unit 8: the centreline, MU_FRACTION 0.56
line_race = race(UNIT_09)  # Unit 9: the racing line, MU_FRACTION 0.54
for name, result in (("Unit 8, the centreline, f = 0.56", centre_race), ("Unit 9, the racing line, f = 0.54", line_race)):
    print(f"{name}: {result.total_time:.2f} s, {result.collisions} collisions, finished: {result.finished}")
assert centre_race.finished and centre_race.collisions == 0 and abs(centre_race.total_time - 301.95) <= TICK
assert line_race.finished and line_race.collisions == 0 and abs(line_race.total_time - 279.60) <= TICK

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, result in (("Unit 8: the centreline", centre_race), ("Unit 9: the racing line", line_race)):
    first_lap = result.telemetry["t"] < 95.0
    ax.plot(result.telemetry["t"][first_lap], result.telemetry["offset"][first_lap], label=name)
ax.set_xlabel("t (s)")
ax.set_ylabel("offset from the centreline (m, left +)")
ax.set_title("test-circuit, seed 0: where across the road the car drives")
ax.legend()
plt.show()

Both finish with 0 collisions, and the racing line is **22.35 s** faster on this seed.
The plot shows how: Unit 8's car stays within about 1.5 m of the centreline, while Unit 9's swings 3 to 4 m to either side, using the road.

But look at the other number.
Unit 9 ships `MU_FRACTION = 0.54`, one grid step **below** Unit 8's 0.56.
The new line broke the speed tuning: the probe record of plan `roar-008` (seeds 0–4 on each of the five training tracks; "not clean" = not clean on at least one seed of the track named) is

| `MU_FRACTION` f | 0.56 | 0.58 | 0.60 |
|---|---|---|---|
| the centreline (Unit 8) | clean | clean: **the edge** | not clean on `test-circuit-seed4` |
| the racing line | clean: **the edge** | not clean on `test-circuit-seed4` | not clean on `test-circuit-seed4` |

With the line, Unit 8's frozen 0.56 still finishes clean on all five training tracks, but its **margin is gone**: it now sits on the edge, and 0.58, the value Unit 8 proved was the centreline's edge, crashes on `test-circuit-seed4`.
The line and the speed plan are one thing: change the line and you must re-tune the speed.

This unit builds the line (§1–§4), shows a trap in building it (§5), follows it (§6), and then measures it honestly: at the same f (§7), re-tuned (§8), against a cheaper idea (§9).

## 1. A line as offsets along the normals

A line is a choice, at every waypoint i, of how far to move **sideways**: a distance n_i along the waypoint's left unit normal N_i (left positive), so the line's point is

$$p'_i = p_i + n_i\,N_i .$$

The normal is the direction of the chord from waypoint i − 1 to waypoint i + 1, turned 90° to the left: for a chord (d_x, d_y) of length h it is (−d_y / h, d_x / h).
The lap is a loop, so the ends wrap round (`np.roll`, as Unit 7's curvature did).

The road limits the choice.
racekit's tracks carry a **half-width** w_i at every waypoint, and the line keeps a margin of 2.5 m from each edge (the car is about 2 m wide, and it does not follow its line perfectly):

$$|n_i| \le w_i - 2.5 .$$

In [ ]:
track = load_track("test-circuit")
x, y, n = track.x, track.y, len(track)


def menger(x, y):
    # Unit 2's signed Menger curvature at every point of a closed polyline (left turns positive).
    xa, ya, xc, yc = np.roll(x, 1), np.roll(y, 1), np.roll(x, -1), np.roll(y, -1)
    twice_area = (x - xa) * (yc - ya) - (y - ya) * (xc - xa)
    sides = np.hypot(x - xa, y - ya) * np.hypot(xc - x, yc - y) * np.hypot(xc - xa, yc - ya)
    return 2.0 * twice_area / sides


def normals(x, y):
    # The left unit normal at every waypoint: the chord from i - 1 to i + 1, turned 90 degrees left.
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


nx, ny = normals(x, y)
chord_x, chord_y = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
assert np.max(np.abs(np.hypot(nx, ny) - 1.0)) < 1e-12  # unit length
assert np.max(np.abs(nx * chord_x + ny * chord_y)) < 1e-9  # at right angles to the chord
room = track.half_width - 2.5
print(f"{n} waypoints; half-width {track.half_width.min():.1f} to {track.half_width.max():.1f} m, "
      f"so each offset may be {room.min():.1f} to {room.max():.1f} m either side")

tightest = int(np.argmax(np.abs(menger(x, y))))
near = np.arange(tightest - 60, tightest + 61) % n
fig, ax = plt.subplots(figsize=(7, 6))
for side in (1.0, -1.0):
    ax.plot(x[near] + side * track.half_width[near] * nx[near], y[near] + side * track.half_width[near] * ny[near],
            color="black")
    ax.plot(x[near] + side * room[near] * nx[near], y[near] + side * room[near] * ny[near], color="grey",
            linestyle="--")
ax.plot(x[near], y[near], color="tab:blue", label="the centreline")
ax.plot([], [], color="black", label="the edges")
ax.plot([], [], color="grey", linestyle="--", label="2.5 m inside the edges")
ax.set_aspect("equal")
ax.set_title(f"test-circuit: the tightest corner (waypoint {tightest}) and the room a line has")
ax.legend()
plt.show()

Every waypoint has between 3.0 and 3.5 m of room on either side.

So the unknowns are n_0 … n_{n−1}, one per waypoint: 1331 numbers on `test-circuit`, and more than 2800 on `monza-like`.
Solving for that many free offsets is slow and badly conditioned (a probe of plan `roar-008` could not solve a full-resolution curvature objective in 500 s), and neighbouring offsets that are free to differ can zig-zag.
The next section asks for far fewer numbers, chosen so that the line is smooth **by construction**.

## 2. The reduced problem: a cubic B-spline of offsets

Choose m = ⌊n / 10⌋ **control values** c_0 … c_{m−1}, one per 10 waypoints (133 on `test-circuit`), and make every offset a **weighted average** of four neighbouring control values.
Waypoint i sits at t = i·m / n on the control values' scale; with j = ⌊t⌋ and u = t − j (between 0 and 1),

$$n_i = w_0(u)\,c_{j-1} + w_1(u)\,c_j + w_2(u)\,c_{j+1} + w_3(u)\,c_{j+2}\quad(\text{indices mod } m),$$

$$w_0 = \tfrac{(1-u)^3}{6},\quad w_1 = \tfrac{3u^3 - 6u^2 + 4}{6},\quad w_2 = \tfrac{-3u^3 + 3u^2 + 3u + 1}{6},\quad w_3 = \tfrac{u^3}{6}.$$

This is the **periodic uniform cubic B-spline**.
Three facts about the weights do all the work:

- they are **never negative** and they **sum to 1**, for every u, so n_i is a weighted average of four control values, and |n_i| is at most the largest of their sizes;
- they are cubic in u and join smoothly from one piece to the next (the curve and its first two derivatives are continuous), so the offsets have no corners;
- every offset is **linear** in c: stacking the weights into an n × m matrix B (row i holds waypoint i's four weights), the offsets are n = B c.

The first fact is how the road's limits become limits on the control values: if every control value that waypoint i averages satisfies |c_j| ≤ w_i − 2.5, then so does n_i.

In [ ]:
STEP = 10  # waypoints per control value


def spline_basis(n, step=STEP):
    # B[i, j]: the weight of control value j in waypoint i's offset (four weights a row, summing to 1).
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


B = spline_basis(n)
m = B.shape[1]
print(f"B is {n} x {m}; the smallest weight is {np.min(B):.1f}; "
      f"the rows sum to 1 within {np.max(np.abs(B.sum(axis=1) - 1)):.0e}")
assert np.min(B) >= 0 and np.max(np.abs(B.sum(axis=1) - 1)) < 1e-12

# agents/unit-09.py rebuilds its offsets from its control values the same way, in its __init__ (numpy only)
with shim_installed():
    Unit09, _ = load_submission(UNIT_09)
reference_agent = Unit09(track.waypoints, None)
print("the reference's offsets:", f"{min(reference_agent.offsets):.2f} to {max(reference_agent.offsets):.2f} m")

The spline has 133 unknowns instead of 1331, and whatever their values, the line they make is smooth.
The reference's offsets reach ±3.5 m, the whole of the room on the widest stretches.
Where do its control values come from?

## 3. The smoothest line, as bounded least squares

What should the line minimise?
The fast line is the one the car can take fastest, and the cap of Unit 6 says the speed through a corner grows as 1 / √|κ|: a line with **less curvature** is a faster line.
Curvature itself is a nonlinear function of the offsets, so we use the usual stand-in for it, the **second difference** of the line's points,

$$\Delta^2 p'_i = p'_{i+1} - 2\,p'_i + p'_{i-1},$$

which, for points about Δs apart, is roughly κ_i·Δs² long and points into the turn.
The **smoothest line** minimises the sum of their squared lengths, x and y together:

$$\sum_i \lVert \Delta^2 p'_i \rVert^2 = \sum_i \left(\Delta^2 x'_i\right)^2 + \left(\Delta^2 y'_i\right)^2 .$$

With p'_i = p_i + n_i N_i and n = B c, the x-part is Δ²x'_i = Δ²x_i + Δ²(N_x B c)_i, which is **linear in c**: a column of the matrix D(N_x B) (the second difference of each column of N_x B) for every control value, plus a constant.
So the objective is ‖A c − b‖² with

$$A = \begin{bmatrix} D(N_x B) \\ D(N_y B) \end{bmatrix} \ (2n \times m), \qquad b = -\begin{bmatrix} \Delta^2 x \\ \Delta^2 y \end{bmatrix}:$$

**linear least squares**, Unit 8's tool.
Try it without any limits first:

In [ ]:
def second_difference(values):
    # v[i + 1] - 2 v[i] + v[i - 1] down the rows, round the loop (a 2-D array: every column at once).
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


A = np.vstack([second_difference(nx[:, None] * B), second_difference(ny[:, None] * B)])
b = -np.concatenate([second_difference(x), second_difference(y)])
print(f"A is {A.shape[0]} x {A.shape[1]}, b has {b.size} entries")

free_c = np.linalg.lstsq(A, b, rcond=None)[0]
free_offsets = B @ free_c
print(f"the unbounded least-squares line: offsets up to {np.max(np.abs(free_offsets)):.1f} m from the centreline; "
      f"{int(np.sum(np.abs(free_offsets) > room))} of {n} waypoints are outside the room")
assert np.max(np.abs(free_offsets)) > 10.0

Unbounded, the smoothest line is **tens of metres off the road**: it cuts across the infield.
The road's limits have to be part of the problem.

**The bounds.**
Control value j appears (with a positive weight) in the offsets of the waypoints of its **support**, about 40 waypoints around it.
Give it room only as far as **every** waypoint in its support allows,

$$|c_j| \le \min_{i\,:\,B_{ij} > 0} (w_i - 2.5),$$

and then, by §2, every waypoint is inside the margin, by construction.

**The start pin.**
The car starts on the centreline, at waypoint 0.
If the line starts 3 m to one side, the car is 3 m off its own line on the very first tick, and its first seconds are spent correcting that (a probe of plan `roar-008` without the pin started 2.94 m off its line).
So every control value whose support touches the 30 waypoints either side of the start is **0**, and the line is exactly the centreline there.

That is **bounded linear least squares**: minimise ‖A c − b‖² subject to lower ≤ c ≤ upper.
`scipy.optimize.lsq_linear` solves it; `method="bvls"` (bounded-variable least squares) is an exact active-set method for a problem of this size.
It needs every lower bound strictly below its upper bound, so a pinned value gets the bounds ±1e-12 m instead of exactly 0.

**Making it fast: QR first.**
Unit 8 showed that with A = Q R (Q has orthonormal columns, R is m × m and upper triangular),

$$\lVert A c - b \rVert^2 = \lVert R c - Q^{\mathsf T} b \rVert^2 + \text{const},$$

the constant being the part of b outside A's column space, which no c can change.
So the bounded problem on the 2662 × 133 matrix A has the same solution as the one on the 133 × 133 matrix R.
A probe of plan `roar-008` gave `lsq_linear` the full A: 11–16 s per `test-circuit` track and 110 s on `monza-like`; on R it gives the same line to 1e-12 m in 3–4 s and 16 s.

In [ ]:
from scipy.optimize import lsq_linear

MARGIN = 2.5  # m kept from each edge
PIN = 30  # waypoints either side of the start held on the centreline


def control_bounds(basis, half_width, margin=MARGIN, pin=PIN):
    # The largest |c_j| every waypoint in control value j's support allows; 0 if the support touches the pin.
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(half_width[support] - margin)
    return high


def solve_line(track, margin=MARGIN, pin=PIN):
    # The smoothest line's control values: QR, then bounded least squares on R c = Q^T b.
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    high = control_bounds(basis, track.half_width, margin, pin)
    Q, R = np.linalg.qr(A)
    result = lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls")
    return result.x


control = solve_line(track)
offsets = B @ control
high = control_bounds(B, track.half_width)
print(f"{m} control values, {int(np.sum(high == 0))} pinned at the start, "
      f"{int(np.sum((high > 0) & (np.abs(np.abs(control) - high) < 1e-9)))} more on a bound")
print(f"the worst waypoint is {np.max(np.abs(offsets) - room):.0e} m outside its room; "
      f"the largest offset within 30 waypoints of the start is {max(np.max(np.abs(offsets[:PIN])), np.max(np.abs(offsets[n - PIN:]))):.0e} m")
gap = np.max(np.abs(offsets - np.array(reference_agent.offsets)))
print(f"the largest difference from the line agents/unit-09.py ships: {gap:.0e} m")
assert np.max(np.abs(offsets) - room) <= 1e-9  # U9.1: inside the margin at every waypoint
assert np.max(np.abs(offsets[:PIN])) <= 1e-9 and np.max(np.abs(offsets[n - PIN:])) <= 1e-9  # U9.1: the pin
assert gap <= 1e-6  # U9.1: the shipped LINE_CONTROL re-solves

Besides the 9 pinned control values, the solver puts 43 **on a bound**: the line uses all the room it is allowed, which is what a racing line does.
No waypoint is outside the margin (the 1e-12 is the solver's slack), the line is the centreline at the start, and it is the line `agents/unit-09.py` ships, to the last bit.

## 4. What the line changes

Three numbers describe a closed line: the **total squared curvature** Σ κ_i² Δs_i (Δs_i the segment from i to i + 1), which is what the smoothness objective stands in for; the **lap length**; and the **tightest radius**, 1 / max|κ_i|, which sets the slowest corner.

In [ ]:
def line_summary(x, y):
    # (total squared curvature sum kappa^2 ds, lap length in m, tightest radius in m) of a closed line.
    kappa = menger(x, y)
    ds = np.hypot(np.roll(x, -1) - x, np.roll(y, -1) - y)
    return float(np.sum(kappa ** 2 * ds)), float(ds.sum()), float(1.0 / np.max(np.abs(kappa)))


line_x, line_y = x + offsets * nx, y + offsets * ny
centre_numbers, line_numbers = line_summary(x, y), line_summary(line_x, line_y)
for name, (total, length, radius) in (("the centreline", centre_numbers), ("the racing line", line_numbers)):
    print(f"{name:16}: sum kappa^2 ds {total:.4f} 1/m, lap {length:.1f} m, tightest radius {radius:.1f} m")
change = line_numbers[0] / centre_numbers[0] - 1
print(f"total squared curvature {change:+.0%}, lap {line_numbers[1] - centre_numbers[1]:+.1f} m")
assert abs(centre_numbers[0] - 0.2880) <= 5e-5 and abs(line_numbers[0] - 0.2326) <= 0.002  # U9.1
assert abs(centre_numbers[1] - 2661.7) <= 0.05 and abs(line_numbers[1] - 2636.2) <= 0.05
assert abs(centre_numbers[2] - 18.0) <= 0.05 and abs(line_numbers[2] - 15.2) <= 0.05

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(track.s, np.abs(menger(x, y)), color="lightgrey", label="the centreline")
ax.plot(track.s, np.abs(menger(line_x, line_y)), label="the racing line")
ax.set_xlabel("distance along the centreline (m)")
ax.set_ylabel("|curvature| (1/m)")
ax.set_title("test-circuit: the curvature of the two lines")
ax.legend()
plt.show()

The line lowers the total squared curvature by **19%** (0.2880 → 0.2326) and shortens the lap by 25.6 m (2661.7 → 2636.2 m).
But it does **not** make every corner gentler: the tightest point gets **tighter**, 18.0 m → 15.2 m.
The objective is a sum over the whole lap, so it happily trades one corner for the rest: a little extra curvature at one apex buys straighter sections around it.
On `monza-like` (probe record, plan `roar-008` U9.1) the trade goes the other way: −41% (0.2421 → 0.1428), a shorter lap (5714.5 → 5684.7 m), and a tightest radius that is wider (25.0 → 28.3 m).

That is why the speed plan must be rebuilt **on the line**: the agent computes Unit 2's Menger curvature at its own line points, Unit 7's profile from it, and Unit 8's cap √(f · μ̂g / |κ|) per section.
Faster almost everywhere, slower at that one corner.

## 5. The trap: joining the control values with straight pieces

The first line of plan `roar-008`'s probes did not use a B-spline.
It put control value j at waypoint ≈ 10 j and **interpolated linearly** between neighbours: n_i = (1 − u) c_j + u c_{j+1}.
That is also linear in c, also a weighted average (two non-negative weights summing to 1), also cheap.
What can go wrong?
Take the reference's own control values and join them with straight pieces instead:

In [ ]:
def linear_basis(n, step=STEP):
    # L[i, j]: waypoint i's offset as the straight-line blend of control values j and j + 1.
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        basis[i, j % m] += 1 - u
        basis[i, (j + 1) % m] += u
    return basis


kinked = linear_basis(n) @ control
kinked_x, kinked_y = x + kinked * nx, y + kinked * ny
kinked_numbers = line_summary(kinked_x, kinked_y)
print(f"joined by straight pieces: sum kappa^2 ds {kinked_numbers[0]:.4f} 1/m (the centreline's {centre_numbers[0]:.4f}), "
      f"tightest radius {kinked_numbers[2]:.1f} m")
assert kinked_numbers[0] > centre_numbers[0] > line_numbers[0]

stretch = np.arange(300, 520)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(stretch, np.abs(menger(kinked_x, kinked_y))[stretch], label="the same control values, straight pieces")
ax.plot(stretch, np.abs(menger(line_x, line_y))[stretch], label="the cubic B-spline")
for j in range(m):
    if stretch[0] <= j * n / m <= stretch[-1]:
        ax.axvline(j * n / m, color="lightgrey", linewidth=0.8)
ax.set_xlabel("waypoint (grey lines: where a control value sits)")
ax.set_ylabel("|curvature| (1/m)")
ax.set_title("test-circuit: straight pieces put a kink at every control value")
ax.legend()
plt.show()

The same control values, joined by straight pieces, have **more** total squared curvature than the centreline, and a tightest radius of 12.0 m.
The plot shows why: at every control value the offset changes slope abruptly, and Menger curvature, measured on three neighbouring points, sees each **kink** as a spike.
The planner sees the spikes too: each one is a corner on its own, with a cap √(f · μ̂g / |κ|) far below what the road needs.
The probe's linearly interpolated line, solved for linear interpolation, still raced (299.35 s at 0.56 on `test-circuit`), but most of the line's gain was lost to the kinks (its Σκ²Δs rose from 0.2880 to 0.3023).

The lesson: the objective (second differences) and the curvature the planner measures (Menger, on three points) are different measures of the same line, and a line that is fine by one can be terrible by the other.
Build lines whose curvature is continuous, and always measure the line the agent will plan on.

## 6. Following a line

The submission does not change its steering law, its speed loop or its planner.
It changes **the points they work on**: `__init__` moves its own copy of the path onto the line, and from then on everything is measured on those points: the curvature, the grip sections and the speed profile, the cross-track error, pure pursuit's target, and the two index searches.

**The `REACHED_RADIUS` trap.**
Unit 1's agent advances its waypoint index when the car comes within `REACHED_RADIUS` = 3 m of the next waypoint.
A car on a line 3.5 m off the centreline never comes within 3 m of the centreline's waypoint there:

In [ ]:
REACHED_RADIUS = 3.0  # m, as in every agent since Unit 1
unreachable = 0
for i in range(n):
    if np.min(np.hypot(x - line_x[i], y - line_y[i])) >= REACHED_RADIUS:
        unreachable += 1
print(f"{unreachable} of {n} line points ({unreachable / n:.0%}) have no centreline waypoint within {REACHED_RADIUS} m")
assert unreachable > 300

# the reference moves its own path onto the line, and searches it
assert np.max(np.abs(np.array(reference_agent.xs) - line_x)) < 1e-9
assert np.max(np.abs(np.array(reference_agent.ys) - line_y)) < 1e-9

A third of the lap.
An agent that searched the centreline's waypoints would stop advancing its index there and steer for a point it has already passed.
So `advance_index` and `nearest_index` search the agent's **own** points, as everything else does.

**The line-relative clean rule.**
Unit 7's clean rule asked for a max |cross-track error| of at most 2 m.
racekit's telemetry `offset` is measured from the **centreline**, so for a line race that rule measures the wrong thing.
The error that matters is the car's distance from **its own line**: at a row with progress p the car is at waypoint (p + 1) mod n (Unit 7), where the line's offset is n_{(p+1) mod n}, so the error is `offset` − n_{(p+1) mod n}.
A line race is **clean** when it finished, with 0 collisions, and that error is at most 2 m on every driving row.
For a centreline race n = 0, and the rule is Unit 7's.

In [ ]:
def line_error(result, offsets):
    # The largest |cross-track error from the race's own line| over the driving rows (held == 0).
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    # The unit's clean rule: finished, 0 collisions, and never more than 2 m from the race's own line.
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def summary(result, offsets):
    verdict = "clean" if clean(result, offsets) else "NOT clean"
    finished = "finished" if result.finished else "did not finish"
    return (f"{verdict}: {result.total_time:.2f} s, {result.collisions} collisions, {finished}, "
            f"{line_error(result, offsets):.2f} m from its line")


centreline = np.zeros(n)
print("Unit 8 (the centreline):", summary(centre_race, centreline))
print("Unit 9 (the racing line):", summary(line_race, offsets))
print(f"Unit 9's max |offset from the centreline|: {line_error(line_race, centreline):.2f} m (Unit 7's rule would say NOT clean)")
assert clean(centre_race, centreline) and clean(line_race, offsets)
assert line_error(line_race, centreline) > CLEAN_ERROR

The racing line's race is 1.52 m from its own line at worst, about as close as Unit 8's car follows the centreline; by Unit 7's rule it would have been "not clean", because the line itself is 3.5 m from the centreline.

## 7. The honest comparison: the same f

The hook compared Unit 8 at 0.56 with Unit 9 at 0.54: two changes at once.
The fair question is what the line alone buys, so race the centreline at the line's f, 0.54:

In [ ]:
centre_054 = race(variant(UNIT_08, "centre-0.54", MU_FRACTION=0.54))
print("the centreline, f = 0.54:", summary(centre_054, centreline))
print("the racing line, f = 0.54:", summary(line_race, offsets))
print(f"the line alone buys {centre_054.total_time - line_race.total_time:.2f} s on seed 0")
assert clean(centre_054, centreline) and abs(centre_054.total_time - 304.55) <= TICK

**About 25 s at the same f.**
Over seeds 0–4 (probe record, plan `roar-008` U9.2, all clean; the milestone and a routed test re-race them):

| track, f | the centreline | local widening (§9) | the racing line |
|---|---|---|---|
| `test-circuit`, 0.54 | 304.70 s | not clean | **279.60 s** |
| `monza-like`, 0.60 | 500.70 s | 487.55 s | **473.45 s** |

The spreads are at most 0.25 s, so these are differences of 100 times the noise: a **noise-aware** win (Unit 1) on both tracks.

## 8. A new line breaks the tuning: re-tune by the margin rule

Now the break the hook promised.
Unit 8 found the centreline's family edge at 0.58 (clean on all five training tracks; 0.60 crashes on `test-circuit-seed4`).
Solve `test-circuit-seed4`'s own line, write it into the reference with that track's measured grip (exactly as Checkpoint A's settings were written in), and race it at 0.58 and at 0.56:

In [ ]:
def line_variant(track_name, control, fraction, name):
    # agents/unit-09.py with this track's line, its measured grip and a planner fraction.
    return variant(UNIT_09, name, LINE_CONTROL=tuple(float(value) for value in control),
                   MU_G=MEASURED_MU_G[track_name], MU_FRACTION=fraction)


seed4 = load_track("test-circuit-seed4")
seed4_control = solve_line(seed4)
seed4_offsets = spline_basis(len(seed4)) @ seed4_control
at_058 = race(line_variant("test-circuit-seed4", seed4_control, 0.58, "seed4-line-0.58"), track="test-circuit-seed4")
at_056 = race(line_variant("test-circuit-seed4", seed4_control, 0.56, "seed4-line-0.56"), track="test-circuit-seed4")
print("test-circuit-seed4, the line, f = 0.58:", summary(at_058, seed4_offsets))
print("test-circuit-seed4, the line, f = 0.56:", summary(at_056, seed4_offsets))
assert not clean(at_058, seed4_offsets)  # U9.3: Unit 8's edge value is not clean with the line
assert clean(at_056, seed4_offsets) and abs(at_056.total_time - 273.40) <= TICK

At 0.58 the line race collides and never finishes; at 0.56 it is clean.
Over seeds 0–4 on the five training tracks (probe record, plan `roar-008` U9.3–U9.4, medians; "no" = not clean on at least one seed):

| f, the racing line | 0.54 | 0.56 | 0.58 | 0.60 |
|---|---|---|---|---|
| `test-circuit` | 279.60 | 277.10 | 274.65 | 272.30 |
| `test-circuit-seed1` | 276.20 | 273.80 | 271.50 | 269.30 |
| `test-circuit-seed2` | 275.50 | 273.05 | 270.80 | 268.65 |
| `test-circuit-seed3` | 275.65 | 273.25 | 270.90 | 268.70 |
| `test-circuit-seed4` | 275.95 | 273.45 | no | no |

The line's family edge is **0.56**, one grid step below the centreline's 0.58.
Why lower?
The line's corners are taken faster and closer to the edge of the road, and the line itself asks the steering to swing the car across the road, so the same fraction of the grip leaves less room for the follower's errors.

**The margin rule** (Unit 7) steps one grid step inside the edge: the line freezes **`MU_FRACTION = 0.54`**.
Unit 8's 0.56 is still clean, but it is now the edge itself; it had a margin on the centreline and it has none on the line.

**The transfer.**
The frozen value was tuned on the training family only; the test is the unseen `test-circuit-seed2026`, raced once per seed, as in Checkpoint A:

In [ ]:
seed2026 = load_track("test-circuit-seed2026")
seed2026_control = solve_line(seed2026)
seed2026_offsets = spline_basis(len(seed2026)) @ seed2026_control
frozen = race(line_variant("test-circuit-seed2026", seed2026_control, 0.54, "seed2026-line-0.54"),
              track="test-circuit-seed2026")
edge = race(line_variant("test-circuit-seed2026", seed2026_control, 0.58, "seed2026-line-0.58"),
            track="test-circuit-seed2026")
print("test-circuit-seed2026, the line, f = 0.54:", summary(frozen, seed2026_offsets))
print("test-circuit-seed2026, the line, f = 0.58:", summary(edge, seed2026_offsets))
assert clean(frozen, seed2026_offsets) and abs(frozen.total_time - 282.10) <= TICK
assert edge.finished and edge.collisions == 0 and not clean(edge, seed2026_offsets)  # U9.4: 0.58 is not clean

The frozen 0.54 is clean on the unseen track, **282.05 s** over seeds 0–4 (probe record, plan `roar-008` U9.4), against Unit 8's 303.30 s there and Checkpoint A's 310.80 s.
At 0.58 the car finishes without touching a wall, but on seed 0 it is **2.34 m from its line** (the probe saw up to 2.39 m over seeds 0–4): a lap that sloppy is a crash waiting for a slightly different car, which is exactly why the clean rule asks for more than "no collision".

## 9. Local widening: the cheaper idea, measured

A global line moves the car everywhere.
A cheaper idea, closer to what the winning teams of the real race did, is to **widen only the tight corners**: keep the centreline, except in a window around every point tighter than 60 m radius, where the line may move.
In the solver that is one more set of pinned control values: every control value whose support leaves the windows (60 waypoints either side of a tight point) is 0.

In [ ]:
LOCAL_RADIUS = 60.0  # m: open a window around every centreline point tighter than this
LOCAL_WINDOW = 60  # waypoints either side of a tight point


def local_line(track, radius=LOCAL_RADIUS, window=LOCAL_WINDOW, margin=MARGIN, pin=PIN):
    # The smoothest line that may leave the centreline only inside the windows around the tight points.
    x, y = track.x, track.y
    count = len(x)
    near = np.zeros(count, dtype=bool)
    for i in np.nonzero(np.abs(menger(x, y)) > 1.0 / radius)[0]:
        near[np.arange(i - window, i + window + 1) % count] = True
    nx, ny = normals(x, y)
    basis = spline_basis(count)
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    high = control_bounds(basis, track.half_width, margin, pin)
    for j in range(basis.shape[1]):
        if np.sum(~near[basis[:, j] > 0]) > 0:
            high[j] = 0.0
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


local_control = local_line(track)
local_offsets = B @ local_control
local_numbers = line_summary(x + local_offsets * nx, y + local_offsets * ny)
print(f"local widening: sum kappa^2 ds {local_numbers[0]:.4f} 1/m, lap {local_numbers[1]:.1f} m, "
      f"tightest radius {local_numbers[2]:.1f} m")
local_race = race(variant(UNIT_09, "local-0.50", LINE_CONTROL=tuple(float(value) for value in local_control),
                          MU_FRACTION=0.50))
print("local widening, f = 0.50:", summary(local_race, local_offsets))
assert not clean(local_race, local_offsets)  # U9.5: not clean even at 0.50

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(track.s, offsets, label="the global line")
ax.plot(track.s, local_offsets, label="local widening")
ax.set_xlabel("distance along the centreline (m)")
ax.set_ylabel("offset n (m, left +)")
ax.set_title("test-circuit: the two lines' offsets")
ax.legend()
plt.show()

Local widening, as built here, is **not clean** on `test-circuit` even at 0.50, two grid steps below the global line's frozen value (probe record, plan `roar-008` U9.5: not clean at any raced f from 0.50 to 0.60, nor with 120-waypoint windows at 0.50–0.54).
The plot shows why: inside a window the line swings out to use the road, and it must be back at 0 within a few control values of the window's edge.
That swing is sharp, and the follower cannot track it at speed.
On `monza-like`, whose corners are further apart, it is clean and lands between the two (§7's table: 487.55 s, against 473.45 s for the global line and 500.70 s for the centreline).

So the comparison the unit set out to make has an honest answer: on these tracks, the global line beats local widening, and local widening, **as built**, does not even race `test-circuit` clean.
"As built" matters: a probe whose control values were interpolated linearly (§5) had a local widening that was clean at 0.56, and a window that is not symmetric is a Challenge exercise.

## 10. The reference: `agents/unit-09.py`

The submission is Unit 8's file with three changes:

- `LINE_STEP = 10` and `LINE_CONTROL`, the 133 control values of `test-circuit`'s line, solved by §3's solver and written at full precision, or `None` for the centreline;
- `__init__` evaluates the B-spline at every waypoint (numpy only: a submission may not import `scipy`, so the line is **solved offline**, here, and shipped as constants), moves its own path points along the normals, and works on those points from then on;
- `MU_FRACTION = 0.54`, re-tuned for the line by the margin rule.

`LINE_CONTROL` and `MU_G` are per track: on another track, write that track's values in with `write_variant`, as §8 did.
With `LINE_CONTROL = None` the file must race **exactly** like Unit 8's at 0.54, tick for tick, because nothing else changed:

In [ ]:
no_line = race(variant(UNIT_09, "no-line", LINE_CONTROL=None))
same = all(np.sum(no_line.telemetry[column] != centre_054.telemetry[column]) == 0 for column in ("t", "x", "y", "v"))
print(f"LINE_CONTROL = None: {no_line.total_time:.2f} s; Unit 8 at 0.54: {centre_054.total_time:.2f} s; "
      f"identical telemetry: {same}")
assert same and no_line.total_time == centre_054.total_time  # U9.6

Over seeds 0–4 the reference is clean on `test-circuit` at **279.60 s** (plan `roar-008` U9.6), against 301.95 s for Unit 8 and 304.15 s for Unit 7.

What this unit adds to the toolbox:

- a **reduced problem** (a few control values instead of a value per waypoint) whose structure, a weighted average, turns limits on the road into limits on the unknowns;
- a nonlinear goal (low curvature) replaced by a **linear least-squares stand-in**, solved with QR and bounds;
- the **measured** check that the stand-in did what was wanted (Σκ²Δs, the tightest radius, and Menger curvature on the points the planner will use);
- a clean rule measured against **what the agent is trying to do**, not a fixed reference;
- and the habit that a change to the line is a change to the speed plan: **re-tune**, by the margin rule, and check the transfer.

Next unit: model predictive control, "plan the next 2 s, every tick", where the agent uses Unit 3's model to choose its actions instead of following a fixed line and profile.